# 🚗 Design & Security Analysis of Self-Driving Car System

## How to Run
1. Install dependencies (Cell 1)
2. Create the Streamlit app file (Cell 2)
3. Launch the app (Cell 3)

---
**Features:**
- Real-time simulated sensor data (LIDAR, Camera, GPS, Radar, IMU, CAN Bus)
- Cyber attack simulation on 6 attack vectors
- Animated physical system impact visualization
- Live attack propagation map
- Security threat dashboard

In [ ]:
# Cell 1: Install required packages
import subprocess, sys

packages = [
    'streamlit',
    'plotly',
    'pandas',
    'numpy',
    'matplotlib'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed successfully!')

In [ ]:
# Cell 2: Write the Streamlit app to app.py

app_code = '''
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import time
import random
import math

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Self-Driving Car Security Analysis",
    page_icon="🚗",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ─────────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────────
st.markdown(\'\'\'<style>
@import url(\'https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700;900&family=Share+Tech+Mono&display=swap\');

html, body, [class*="css"] {
    font-family: \'Share Tech Mono\', monospace;
    background-color: #0a0e1a;
    color: #c8d8e8;
}
.stApp { background: linear-gradient(135deg, #0a0e1a 0%, #0d1b2a 50%, #0a0e1a 100%); }

h1, h2, h3 { font-family: \'Orbitron\', sans-serif !important; color: #00d4ff !important; }

.attack-card {
    background: linear-gradient(135deg, rgba(255,50,50,0.15), rgba(255,50,50,0.05));
    border: 1px solid rgba(255,80,80,0.5);
    border-radius: 10px;
    padding: 15px;
    margin: 8px 0;
    animation: pulse-red 2s infinite;
}
.safe-card {
    background: linear-gradient(135deg, rgba(0,212,100,0.15), rgba(0,212,100,0.05));
    border: 1px solid rgba(0,212,100,0.4);
    border-radius: 10px;
    padding: 15px;
    margin: 8px 0;
}
.metric-box {
    background: rgba(0,212,255,0.08);
    border: 1px solid rgba(0,212,255,0.3);
    border-radius: 8px;
    padding: 12px;
    text-align: center;
}
@keyframes pulse-red {
    0%,100% { box-shadow: 0 0 5px rgba(255,50,50,0.3); }
    50% { box-shadow: 0 0 20px rgba(255,50,50,0.8); }
}
@keyframes pulse-blue {
    0%,100% { box-shadow: 0 0 5px rgba(0,212,255,0.3); }
    50% { box-shadow: 0 0 20px rgba(0,212,255,0.8); }
}
.stButton>button {
    background: linear-gradient(90deg, #ff3232, #ff6b00);
    color: white;
    border: none;
    border-radius: 6px;
    font-family: \'Orbitron\', sans-serif;
    font-weight: 700;
    letter-spacing: 1px;
    padding: 10px 24px;
    transition: all 0.3s;
}
.stButton>button:hover { transform: scale(1.05); box-shadow: 0 0 20px rgba(255,50,50,0.6); }
.stSelectbox>div>div { background: rgba(0,20,40,0.9) !important; border: 1px solid #00d4ff !important; }
div[data-testid="stMetricValue"] { color: #00d4ff !important; font-family: \'Orbitron\',sans-serif; font-size:1.8rem; }
div[data-testid="stMetricLabel"] { color: #8ab4cc !important; }
</style>\'\'\', unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SYSTEM DEFINITIONS
# ─────────────────────────────────────────────
CYBER_ATTACKS = {
    "GPS Spoofing": {
        "icon": "📡",
        "color": "#ff4444",
        "physical_impacts": ["GPS/Navigation", "Route Planning", "Steering System"],
        "description": "Attacker sends fake GPS signals to mislead the vehicle location.",
        "consequences": [
            "Vehicle navigates to wrong location",
            "Steering commands deviate from intended path",
            "Vehicle may enter restricted/dangerous zones"
        ],
        "severity": "CRITICAL",
        "sensor": "GPS Receiver"
    },
    "CAN Bus Injection": {
        "icon": "🔌",
        "color": "#ff6600",
        "physical_impacts": ["Braking System", "Engine Control", "Transmission"],
        "description": "Malicious frames injected into CAN bus to control vehicle actuators.",
        "consequences": [
            "Sudden unintended braking or acceleration",
            "Engine RPM manipulation",
            "Gear shift interference at high speed"
        ],
        "severity": "CRITICAL",
        "sensor": "CAN Bus Interface"
    },
    "LIDAR Spoofing": {
        "icon": "🔦",
        "color": "#cc00ff",
        "physical_impacts": ["Obstacle Detection", "Braking System", "Steering System"],
        "description": "Laser pulses injected to create fake obstacles or blind the LIDAR.",
        "consequences": [
            "Emergency braking for non-existent obstacles",
            "Failure to detect real pedestrians/vehicles",
            "Erratic steering to avoid phantom objects"
        ],
        "severity": "HIGH",
        "sensor": "LIDAR Sensor"
    },
    "Camera Adversarial Attack": {
        "icon": "📷",
        "color": "#ffaa00",
        "physical_impacts": ["Vision System", "Traffic Sign Recognition", "Lane Keeping"],
        "description": "Adversarial patches placed on signs/roads fool the vision neural network.",
        "consequences": [
            "Stop sign misclassified as speed limit sign",
            "Lane markings ignored — vehicle drifts",
            "Traffic light state misread"
        ],
        "severity": "HIGH",
        "sensor": "Camera Array"
    },
    "Radar Jamming": {
        "icon": "📶",
        "color": "#00aaff",
        "physical_impacts": ["Adaptive Cruise Control", "Collision Avoidance", "Braking System"],
        "description": "High-power RF signal jams radar receiver, blinding it to nearby vehicles.",
        "consequences": [
            "ACC fails to slow for slower vehicle ahead",
            "Rear-end collision risk increases significantly",
            "Emergency braking system disabled"
        ],
        "severity": "HIGH",
        "sensor": "Radar Sensor"
    },
    "V2X / OTA Attack": {
        "icon": "📲",
        "color": "#00ffaa",
        "physical_impacts": ["Software/ECU", "All Physical Systems", "Remote Control"],
        "description": "Man-in-the-middle on V2X comms or OTA update channel injects malicious firmware.",
        "consequences": [
            "Full vehicle takeover via remote commands",
            "Persistent malware across all ECUs",
            "Complete loss of driver/system control"
        ],
        "severity": "CRITICAL",
        "sensor": "V2X / Telematics Unit"
    }
}

PHYSICAL_SYSTEMS = [
    "GPS/Navigation", "Braking System", "Engine Control",
    "Steering System", "Vision System", "Obstacle Detection",
    "Adaptive Cruise Control", "Software/ECU", "Transmission",
    "Route Planning", "Lane Keeping", "Traffic Sign Recognition",
    "Collision Avoidance", "Remote Control", "All Physical Systems"
]

# ─────────────────────────────────────────────
# SIMULATED SENSOR DATA GENERATOR
# ─────────────────────────────────────────────
def generate_sensor_data(attack_active=False, attack_type=None, t=0):
    base = {
        "speed_kmh": 60 + 5*math.sin(t*0.3) + random.gauss(0,1),
        "gps_lat": 19.2183 + 0.0001*math.sin(t*0.05) + random.gauss(0,0.00005),
        "gps_lon": 72.9781 + 0.0001*math.cos(t*0.05) + random.gauss(0,0.00005),
        "gps_accuracy_m": 1.5 + random.gauss(0,0.2),
        "lidar_front_m": 25 + 3*math.sin(t*0.2) + random.gauss(0,0.5),
        "lidar_left_m": 3.5 + random.gauss(0,0.1),
        "lidar_right_m": 3.5 + random.gauss(0,0.1),
        "radar_front_m": 80 + 5*math.cos(t*0.15) + random.gauss(0,1),
        "radar_velocity_rel": -2 + random.gauss(0,0.5),
        "camera_confidence": 0.95 + random.gauss(0,0.02),
        "imu_accel_x": random.gauss(0,0.1),
        "imu_accel_y": random.gauss(0,0.05),
        "imu_yaw_rate": random.gauss(0,0.5),
        "brake_pressure_bar": 0 + abs(random.gauss(0,0.5)),
        "throttle_pct": 28 + random.gauss(0,2),
        "engine_rpm": 1800 + 100*math.sin(t*0.4) + random.gauss(0,20),
        "can_bus_errors": 0,
        "battery_soc_pct": 75 - t*0.01,
    }
    if attack_active and attack_type:
        if attack_type == "GPS Spoofing":
            base["gps_lat"] += random.gauss(0.02, 0.01)
            base["gps_lon"] += random.gauss(0.02, 0.01)
            base["gps_accuracy_m"] = random.uniform(50, 200)
        elif attack_type == "CAN Bus Injection":
            base["brake_pressure_bar"] = random.uniform(80, 150)
            base["throttle_pct"] = random.uniform(80, 100)
            base["engine_rpm"] = random.uniform(5000, 7000)
            base["can_bus_errors"] = random.randint(50, 200)
        elif attack_type == "LIDAR Spoofing":
            base["lidar_front_m"] = random.uniform(0.5, 3)
            base["brake_pressure_bar"] = random.uniform(60, 120)
        elif attack_type == "Camera Adversarial Attack":
            base["camera_confidence"] = random.uniform(0.1, 0.4)
        elif attack_type == "Radar Jamming":
            base["radar_front_m"] = 0
            base["radar_velocity_rel"] = 0
        elif attack_type == "V2X / OTA Attack":
            base["can_bus_errors"] = random.randint(200, 500)
            base["throttle_pct"] = random.uniform(0, 100)
            base["engine_rpm"] = random.uniform(0, 8000)
    return base

# ─────────────────────────────────────────────
# SESSION STATE INIT
# ─────────────────────────────────────────────
if "attack_active" not in st.session_state:
    st.session_state.attack_active = False
if "selected_attack" not in st.session_state:
    st.session_state.selected_attack = list(CYBER_ATTACKS.keys())[0]
if "history" not in st.session_state:
    st.session_state.history = []
if "tick" not in st.session_state:
    st.session_state.tick = 0

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("## ⚙️ CONTROL PANEL")
    st.markdown("---")
    selected = st.selectbox("Select Cyber Attack Vector", list(CYBER_ATTACKS.keys()),
                             index=list(CYBER_ATTACKS.keys()).index(st.session_state.selected_attack))
    st.session_state.selected_attack = selected
    atk = CYBER_ATTACKS[selected]
    st.markdown(f"### {atk[\'icon\']} {selected}")
    sev_color = "#ff3333" if atk[\'severity\'] == "CRITICAL" else "#ffaa00"
    st.markdown(f"<span style=\'color:{sev_color};font-weight:bold;font-size:1.1rem;\'>⚠ Severity: {atk[\'severity\']}</span>", unsafe_allow_html=True)
    st.info(atk[\'description\'])
    st.markdown("**Targeted Sensor:**")
    st.code(atk[\'sensor\'])
    st.markdown("---")
    col1, col2 = st.columns(2)
    with col1:
        if st.button("🔴 LAUNCH ATTACK"):
            st.session_state.attack_active = True
    with col2:
        if st.button("🟢 STOP / RESET"):
            st.session_state.attack_active = False
            st.session_state.history = []
            st.session_state.tick = 0
    st.markdown("---")
    status_text = "🔴 ATTACK IN PROGRESS" if st.session_state.attack_active else "🟢 SYSTEM NORMAL"
    status_color = "#ff3333" if st.session_state.attack_active else "#00ff88"
    st.markdown(f"<div style=\'text-align:center;color:{status_color};font-family:Orbitron,sans-serif;font-weight:700;font-size:1rem;\'>{status_text}</div>",
                unsafe_allow_html=True)
    auto_refresh = st.checkbox("⚡ Live Data Mode (auto-refresh)", value=False)

# ─────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────
st.markdown("<h1 style=\'text-align:center;letter-spacing:3px;font-size:2rem;margin-bottom:0;\'>🚗 SELF-DRIVING CAR SECURITY ANALYSIS</h1>", unsafe_allow_html=True)
st.markdown("<p style=\'text-align:center;color:#5a8a9a;font-size:0.85rem;margin-top:4px;\'>Cyber Attack ➜ Physical System Impact Visualizer</p>", unsafe_allow_html=True)
st.markdown("---")

# ─────────────────────────────────────────────
# GENERATE CURRENT SENSOR DATA
# ─────────────────────────────────────────────
st.session_state.tick += 1
t = st.session_state.tick
sensor_data = generate_sensor_data(st.session_state.attack_active, st.session_state.selected_attack, t)

# Build history (keep last 50 points)
st.session_state.history.append(sensor_data)
if len(st.session_state.history) > 50:
    st.session_state.history = st.session_state.history[-50:]
hist_df = pd.DataFrame(st.session_state.history)

# ─────────────────────────────────────────────
# TOP METRICS ROW
# ─────────────────────────────────────────────
m1, m2, m3, m4, m5, m6 = st.columns(6)
speed_color = "normal"
with m1:
    st.metric("🚗 Speed", f"{sensor_data[\'speed_kmh\']:.1f} km/h")
with m2:
    gps_acc = sensor_data[\'gps_accuracy_m\']
    st.metric("📡 GPS Accuracy", f"{gps_acc:.1f} m", delta="SPOOFED" if gps_acc > 10 else "Normal")
with m3:
    lidar_v = sensor_data[\'lidar_front_m\']
    st.metric("🔦 LIDAR Front", f"{lidar_v:.1f} m", delta="⚠ FAKE OBJ" if lidar_v < 5 else "Clear")
with m4:
    cam_v = sensor_data[\'camera_confidence\']
    st.metric("📷 Cam Confidence", f"{cam_v:.2f}", delta="⚠ LOW" if cam_v < 0.6 else "Good")
with m5:
    can_v = sensor_data[\'can_bus_errors\']
    st.metric("🔌 CAN Errors", f"{int(can_v)}", delta="⚠ INJECTED" if can_v > 10 else "0")
with m6:
    radar_v = sensor_data[\'radar_front_m\']
    st.metric("📶 Radar Front", f"{radar_v:.1f} m", delta="⚠ JAMMED" if radar_v == 0 else "OK")

st.markdown("---")

# ─────────────────────────────────────────────
# MAIN LAYOUT: ATTACK MAP + IMPACT ANIMATION
# ─────────────────────────────────────────────
col_left, col_right = st.columns([1.2, 1])

with col_left:
    st.markdown("### 🗺️ Attack Propagation Map")
    atk_info = CYBER_ATTACKS[st.session_state.selected_attack]
    impacted = atk_info["physical_impacts"]

    # Node positions for the graph
    cyber_nodes = list(CYBER_ATTACKS.keys())
    phys_nodes = PHYSICAL_SYSTEMS

    # Layout: cyber on left column, physical on right column
    cx = [0.05] * len(cyber_nodes)
    cy = [i / (len(cyber_nodes)-1) for i in range(len(cyber_nodes))]
    px2 = [0.95] * len(phys_nodes)
    py2 = [i / (len(phys_nodes)-1) for i in range(len(phys_nodes))]

    fig_map = go.Figure()

    # Draw edges from selected attack to impacted physical systems
    attack_idx = cyber_nodes.index(st.session_state.selected_attack)
    ax_pos = cx[attack_idx]
    ay_pos = cy[attack_idx]
    for ps in impacted:
        if ps in phys_nodes:
            pi = phys_nodes.index(ps)
            fig_map.add_trace(go.Scatter(
                x=[ax_pos, px2[pi]], y=[ay_pos, py2[pi]],
                mode="lines",
                line=dict(color="#ff3333" if st.session_state.attack_active else "#444", width=2.5, dash="dot"),
                showlegend=False, hoverinfo="skip"
            ))

    # Cyber attack nodes
    for i, name in enumerate(cyber_nodes):
        is_sel = (name == st.session_state.selected_attack)
        color = CYBER_ATTACKS[name]["color"] if is_sel else "#334455"
        size = 28 if is_sel else 16
        fig_map.add_trace(go.Scatter(
            x=[cx[i]], y=[cy[i]],
            mode="markers+text",
            marker=dict(size=size, color=color, symbol="diamond",
                        line=dict(width=2, color="white" if is_sel else "#223")),
            text=[CYBER_ATTACKS[name]["icon"]+" "+name],
            textposition="middle right",
            textfont=dict(size=9, color="#c8d8e8"),
            name=name, hoverinfo="name", showlegend=False
        ))

    # Physical system nodes
    for i, ps in enumerate(phys_nodes):
        is_hit = ps in impacted
        color = "#ff5555" if (is_hit and st.session_state.attack_active) else (
                "#ffaa33" if is_hit else "#1a3a4a")
        size = 20 if is_hit else 12
        fig_map.add_trace(go.Scatter(
            x=[px2[i]], y=[py2[i]],
            mode="markers+text",
            marker=dict(size=size, color=color, symbol="circle",
                        line=dict(width=2, color="#ff3333" if (is_hit and st.session_state.attack_active) else "#334")),
            text=[ps],
            textposition="middle left",
            textfont=dict(size=9, color="#ff9999" if (is_hit and st.session_state.attack_active) else "#8ab4cc"),
            name=ps, hoverinfo="name", showlegend=False
        ))

    # Labels
    fig_map.add_annotation(x=0.05, y=1.05, text="⚡ CYBER ATTACKS", showarrow=False,
                            font=dict(color="#00d4ff", size=11, family="Orbitron"),
                            xref="paper", yref="paper")
    fig_map.add_annotation(x=0.95, y=1.05, text="🔧 PHYSICAL SYSTEMS", showarrow=False,
                            font=dict(color="#ff7744", size=11, family="Orbitron"),
                            xref="paper", yref="paper")

    fig_map.update_layout(
        height=480, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,10,20,0.7)",
        xaxis=dict(visible=False, range=[-0.1, 1.5]),
        yaxis=dict(visible=False, range=[-0.05, 1.1]),
        margin=dict(l=0, r=0, t=30, b=0),
        font=dict(color="#c8d8e8")
    )
    st.plotly_chart(fig_map, use_container_width=True)

with col_right:
    st.markdown("### 🚨 Physical Impact Animation")
    # Animated car diagram using SVG-in-plotly
    attack_on = st.session_state.attack_active
    impacted_sys = CYBER_ATTACKS[st.session_state.selected_attack]["physical_impacts"]

    car_parts = {
        "GPS/Navigation":       (0.5, 0.12),
        "Route Planning":       (0.5, 0.05),
        "Vision System":        (0.5, 0.22),
        "Braking System":       (0.18, 0.68),
        "Engine Control":       (0.5, 0.5),
        "Steering System":      (0.5, 0.35),
        "Obstacle Detection":   (0.82, 0.22),
        "Adaptive Cruise Control": (0.18, 0.35),
        "Software/ECU":         (0.82, 0.5),
        "Transmission":         (0.5, 0.65),
        "Lane Keeping":         (0.18, 0.22),
        "Traffic Sign Recognition": (0.82, 0.35),
        "Collision Avoidance":  (0.82, 0.68),
        "Remote Control":       (0.18, 0.5),
        "All Physical Systems": (0.5, 0.5),
    }

    fig_car = go.Figure()

    # Car body shape
    car_body_x = [0.15, 0.85, 0.85, 0.72, 0.28, 0.15, 0.15]
    car_body_y = [0.3, 0.3, 0.75, 0.9, 0.9, 0.75, 0.3]
    fig_car.add_trace(go.Scatter(
        x=car_body_x, y=car_body_y,
        fill="toself",
        fillcolor="rgba(0,50,80,0.6)",
        line=dict(color="#00d4ff", width=2),
        showlegend=False, hoverinfo="skip"
    ))

    # Wheels
    for wx, wy in [(0.2, 0.28), (0.8, 0.28), (0.2, 0.72), (0.8, 0.72)]:
        theta = [i*math.pi/8 for i in range(17)]
        r = 0.07
        fig_car.add_trace(go.Scatter(
            x=[wx + r*math.cos(a) for a in theta],
            y=[wy + r*0.6*math.sin(a) for a in theta],
            fill="toself", fillcolor="#112233",
            line=dict(color="#00aaff", width=1.5),
            showlegend=False, hoverinfo="skip"
        ))

    # System nodes on car
    for sys_name, (sx, sy) in car_parts.items():
        if sys_name == "All Physical Systems":
            continue
        is_hit = sys_name in impacted_sys or "All Physical Systems" in impacted_sys
        node_color = "#ff3333" if (is_hit and attack_on) else (
                     "#ffaa00" if is_hit else "#0a3a5a")
        node_size = 18 if (is_hit and attack_on) else 12
        fig_car.add_trace(go.Scatter(
            x=[sx], y=[sy],
            mode="markers+text",
            marker=dict(size=node_size, color=node_color,
                        symbol="square" if is_hit else "circle",
                        line=dict(width=2, color="white" if (is_hit and attack_on) else "#0a3a5a")),
            text=[sys_name],
            textposition="top center",
            textfont=dict(size=7, color="#ff9999" if (is_hit and attack_on) else "#5a8a9a"),
            name=sys_name, hovertemplate=f"<b>{sys_name}</b><br>Status: {\'⚠ COMPROMISED\' if (is_hit and attack_on) else \'Normal\'}<extra></extra>",
            showlegend=False
        ))

    status_label = "⚠ UNDER ATTACK" if attack_on else "✓ NOMINAL"
    status_col = "#ff3333" if attack_on else "#00ff88"
    fig_car.add_annotation(x=0.5, y=0.97, text=status_label, showarrow=False,
                            font=dict(color=status_col, size=14, family="Orbitron"),
                            xref="paper", yref="paper")

    fig_car.update_layout(
        height=480, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,10,20,0.7)",
        xaxis=dict(visible=False, range=[0, 1]),
        yaxis=dict(visible=False, range=[0, 1]),
        margin=dict(l=0, r=0, t=20, b=0),
        font=dict(color="#c8d8e8")
    )
    st.plotly_chart(fig_car, use_container_width=True)

# ─────────────────────────────────────────────
# CONSEQUENCES & SENSOR CHARTS
# ─────────────────────────────────────────────
st.markdown("---")
col_a, col_b = st.columns([1, 1.5])

with col_a:
    st.markdown("### 💥 Attack Consequences")
    atk_info = CYBER_ATTACKS[st.session_state.selected_attack]
    card_class = "attack-card" if st.session_state.attack_active else "safe-card"
    for c in atk_info["consequences"]:
        icon = "🔴" if st.session_state.attack_active else "🟡"
        st.markdown(f"<div class=\'{card_class}\'>{icon} {c}</div>", unsafe_allow_html=True)

    st.markdown("### 🔧 Impacted Physical Systems")
    for ps in atk_info["physical_impacts"]:
        status = "COMPROMISED ⚠" if st.session_state.attack_active else "AT RISK"
        col_s = "#ff4444" if st.session_state.attack_active else "#ffaa33"
        st.markdown(f"<div style=\'padding:6px 12px;margin:4px 0;border-left:3px solid {col_s};background:rgba(255,100,0,0.07);border-radius:4px;\'>"
                    f"🔧 <b>{ps}</b> — <span style=\'color:{col_s};font-size:0.85rem;\'>{status}</span></div>",
                    unsafe_allow_html=True)

with col_b:
    st.markdown("### 📊 Live Sensor Telemetry")
    if len(hist_df) > 2:
        fig_tel = make_subplots(rows=3, cols=2,
                                subplot_titles=["Speed (km/h)", "GPS Accuracy (m)",
                                                "LIDAR Front (m)", "Camera Confidence",
                                                "CAN Bus Errors", "Engine RPM"],
                                vertical_spacing=0.12, horizontal_spacing=0.1)

        series_map = [
            ("speed_kmh", 1, 1, "#00d4ff"),
            ("gps_accuracy_m", 1, 2, "#ffaa00"),
            ("lidar_front_m", 2, 1, "#cc00ff"),
            ("camera_confidence", 2, 2, "#00ffaa"),
            ("can_bus_errors", 3, 1, "#ff6600"),
            ("engine_rpm", 3, 2, "#00aaff"),
        ]
        for col_name, row, col, color in series_map:
            y_vals = hist_df[col_name].tolist() if col_name in hist_df.columns else []
            fig_tel.add_trace(
                go.Scatter(y=y_vals, mode="lines",
                           line=dict(color=color, width=2),
                           fill="tozeroy",
                           fillcolor=color.replace("#", "rgba(") + ",0.1)",
                           showlegend=False),
                row=row, col=col
            )

        fig_tel.update_layout(
            height=360, paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,10,20,0.8)",
            font=dict(color="#8ab4cc", size=9),
            margin=dict(l=30, r=10, t=30, b=10)
        )
        for i in range(1, 4):
            for j in range(1, 3):
                fig_tel.update_xaxes(showgrid=False, zeroline=False, row=i, col=j)
                fig_tel.update_yaxes(gridcolor="rgba(0,100,150,0.2)", zeroline=False, row=i, col=j)
        st.plotly_chart(fig_tel, use_container_width=True)
    else:
        st.info("Click Refresh or enable Live Data Mode to populate telemetry charts.")

# ─────────────────────────────────────────────
# ALL ATTACKS SEVERITY OVERVIEW
# ─────────────────────────────────────────────
st.markdown("---")
st.markdown("### 🛡️ Security Threat Matrix — All Attack Vectors")

sev_map = {"CRITICAL": 3, "HIGH": 2, "MEDIUM": 1}
atk_names = list(CYBER_ATTACKS.keys())
sev_vals = [sev_map[CYBER_ATTACKS[a]["severity"]] for a in atk_names]
impact_counts = [len(CYBER_ATTACKS[a]["physical_impacts"]) for a in atk_names]
colors_bar = [CYBER_ATTACKS[a]["color"] for a in atk_names]
icons_bar = [CYBER_ATTACKS[a]["icon"] for a in atk_names]

fig_matrix = go.Figure()
fig_matrix.add_trace(go.Bar(
    x=[f"{icons_bar[i]} {atk_names[i]}" for i in range(len(atk_names))],
    y=sev_vals,
    name="Severity Score",
    marker_color=colors_bar,
    text=[CYBER_ATTACKS[a]["severity"] for a in atk_names],
    textposition="outside",
    textfont=dict(color="#c8d8e8", size=11)
))
fig_matrix.add_trace(go.Scatter(
    x=[f"{icons_bar[i]} {atk_names[i]}" for i in range(len(atk_names))],
    y=impact_counts,
    name="# Physical Systems Impacted",
    mode="lines+markers",
    line=dict(color="#00ffaa", width=2, dash="dot"),
    marker=dict(size=10, color="#00ffaa"),
    yaxis="y2"
))
fig_matrix.update_layout(
    height=300,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,10,20,0.8)",
    yaxis=dict(title="Severity (1=Med,2=High,3=Crit)", gridcolor="rgba(0,100,150,0.2)",
               color="#8ab4cc", range=[0, 4]),
    yaxis2=dict(title="# Systems Impacted", overlaying="y", side="right",
                gridcolor="rgba(0,100,150,0.1)", color="#00ffaa", range=[0, 6]),
    legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="#334", font=dict(color="#c8d8e8")),
    font=dict(color="#8ab4cc"),
    margin=dict(l=40, r=50, t=20, b=60),
    xaxis=dict(tickfont=dict(size=9), gridcolor="rgba(0,100,150,0.1)")
)
st.plotly_chart(fig_matrix, use_container_width=True)

# ─────────────────────────────────────────────
# RAW SENSOR TABLE
# ─────────────────────────────────────────────
with st.expander("🔬 Raw Sensor Data (Current Tick)"):
    display_df = pd.DataFrame([sensor_data]).T.rename(columns={0: "Value"})
    display_df["Value"] = display_df["Value"].map(lambda v: f"{v:.4f}" if isinstance(v, float) else v)
    st.dataframe(display_df, use_container_width=True)

# ─────────────────────────────────────────────
# AUTO REFRESH
# ─────────────────────────────────────────────
if auto_refresh:
    time.sleep(1)
    st.rerun()
else:
    if st.button("🔄 Refresh Data"):
        st.rerun()

st.markdown("<p style=\'text-align:center;color:#2a4a5a;font-size:0.75rem;margin-top:30px;\'>Design & Security Analysis of Self-Driving Car System | Simulated Sensor Data</p>",
            unsafe_allow_html=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ app.py written successfully!')
print('📄 File size:', len(app_code), 'characters')

In [ ]:
# Cell 3: Launch the Streamlit app
# This opens the app in your browser

import subprocess
import threading
import time

def run_streamlit():
    subprocess.Popen(
        ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'false'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(3)

print('🚀 Streamlit app launched!')
print('👉 Open your browser at: http://localhost:8501')
print()
print('Controls:')
print('  - Use the sidebar to select an ATTACK VECTOR')
print('  - Click "LAUNCH ATTACK" to simulate the attack')
print('  - Watch the Animation, Propagation Map & Telemetry update')
print('  - Enable "Live Data Mode" for continuous simulation')
print('  - Click "STOP / RESET" to clear and restore normal state')

In [ ]:
# Cell 4 (OPTIONAL): Verify app.py content
with open('app.py', 'r') as f:
    content = f.read()

lines = content.split('\n')
print(f'✅ app.py has {len(lines)} lines')
print(f'   Sections found:')
for keyword in ['generate_sensor_data', 'GPS Spoofing', 'CAN Bus Injection',
                 'LIDAR Spoofing', 'Camera Adversarial', 'Radar Jamming', 
                 'V2X / OTA', 'Attack Propagation', 'Live Sensor', 'Threat Matrix']:
    found = keyword in content
    print(f'   {"✅" if found else "❌"} {keyword}')